# Why Are Deliveries Late? — Part 4: Advanced SQL

Deeper questions on the same data using **CTEs** (`WITH`), **window functions** (`RANK() OVER`, `LAG() OVER`) and `PARTITION BY`.

## Setup

In [1]:
%load_ext sql
%sql sqlite:///data/supply_chain.db
%config SqlMagic.displaylimit = 50

Connecting to 'sqlite:///data/supply_chain.db'

## A1. Which months were worse than average? (CTE)
Build the monthly late rate in a CTE, then return only the months whose late rate is above the overall monthly average.

In [6]:
%%sql
WITH monthly AS (
    SELECT order_month,
           100.0 * COUNT(DISTINCT CASE WHEN is_late = 1 THEN order_id END)
                 / COUNT(DISTINCT order_id) AS late_pct
    FROM order_items
    WHERE is_cancelled=0
    GROUP BY order_month
)
SELECT order_month, ROUND(late_pct, 1) AS late_pct
FROM monthly
WHERE late_pct > (SELECT AVG(late_pct) FROM monthly)
ORDER BY order_month;

Running query in 'sqlite:///data/supply_chain.db'

order_month,late_pct
2015-02-01 00:00:00,57.9
2015-03-01 00:00:00,57.5
2015-05-01 00:00:00,57.5
2015-08-01 00:00:00,57.6
2015-09-01 00:00:00,58.2
2015-10-01 00:00:00,57.4
2016-01-01 00:00:00,58.6
2016-03-01 00:00:00,58.1
2016-05-01 00:00:00,58.0
2016-06-01 00:00:00,59.9


**Finding:** 20 of 37 months were above the 57.3% average, but all months sit in a narrow 55–60% band, so no month is meaningfully worse. The problem is constant, not seasonal.

## A2. Worst countries within each market (RANK + PARTITION BY)
Rank countries by late rate **within each market** (countries with at least 300 shipped orders) and keep the top 3 per market.

In [7]:
%%sql
WITH country_rates AS (
    SELECT market, order_country,
           COUNT(DISTINCT order_id) AS shipped_orders,
           100.0 * COUNT(DISTINCT CASE WHEN is_late = 1 THEN order_id END)
                 / COUNT(DISTINCT order_id) AS late_pct
    FROM order_items
    WHERE is_cancelled = 0
    GROUP BY market, order_country
    HAVING COUNT(DISTINCT order_id) >= 300
),
ranked AS (
    SELECT *,
           RANK() OVER (PARTITION BY market ORDER BY late_pct DESC) AS rnk
    FROM country_rates
)
SELECT market, order_country, shipped_orders, ROUND(late_pct, 1) AS late_pct, rnk
FROM ranked
WHERE rnk <= 3
ORDER BY market, rnk;

Running query in 'sqlite:///data/supply_chain.db'

market,order_country,shipped_orders,late_pct,rnk
Africa,Democratic Republic of the Congo,317,60.9,1
Africa,Egypt,374,58.8,2
Africa,Morocco,360,56.1,3
Europe,Russia,309,59.9,1
Europe,Austria,484,59.3,2
Europe,France,4648,58.4,3
LATAM,Venezuela,302,60.9,1
LATAM,Nicaragua,959,59.6,2
LATAM,El Salvador,1195,57.9,3
Pacific Asia,Thailand,385,58.7,1


**Finding:** Even the worst country in each market is only 2–4 points above the 57.3% average (e.g. Russia 59.9%, Venezuela 60.9%), and these are smaller samples of around 300–400 orders. The largest markets (France, United States) sit right at the average, which confirms the cause is not geographic.

## A3. Month-over-month change (LAG)
For each month, show the late rate, the previous month's late rate, and the change in percentage points.

In [8]:
%%sql
WITH monthly AS (
    SELECT order_month,
           100.0 * COUNT(DISTINCT CASE WHEN is_late = 1 THEN order_id END)
                 / COUNT(DISTINCT order_id) AS late_pct
    FROM order_items
    WHERE is_cancelled = 0
    GROUP BY order_month
)
SELECT order_month,
       ROUND(late_pct, 1) AS late_pct,
       ROUND(LAG(late_pct) OVER (ORDER BY order_month), 1) AS prev_month_pct,
       ROUND(late_pct - LAG(late_pct) OVER (ORDER BY order_month), 1) AS change_pts
FROM monthly
ORDER BY order_month;

Running query in 'sqlite:///data/supply_chain.db'

order_month,late_pct,prev_month_pct,change_pts
2015-01-01 00:00:00,56.2,None,None
2015-02-01 00:00:00,57.9,56.2,1.7
2015-03-01 00:00:00,57.5,57.9,-0.4
2015-04-01 00:00:00,56.6,57.5,-0.9
2015-05-01 00:00:00,57.5,56.6,1.0
2015-06-01 00:00:00,57.0,57.5,-0.5
2015-07-01 00:00:00,56.5,57.0,-0.5
2015-08-01 00:00:00,57.6,56.5,1.1
2015-09-01 00:00:00,58.2,57.6,0.6
2015-10-01 00:00:00,57.4,58.2,-0.8


**Finding:** Month-to-month changes are small, mostly within ±2 percentage points, and the largest drop (−4.4 points in July 2016) reversed the following month. There is no sustained improvement or decline: the late-delivery problem has been stable for three years.